# Stock Market Information System
## Economic Indicators Dashboard

This notebook collects and visualizes key market indicators including:
- Inflation (PCE, CPI)
- Treasury yields (2Y, 10Y, 30Y)
- Liquidity measures (Fed Balance Sheet, M2)
- Market performance (S&P 500)

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install yfinance pandas matplotlib plotly fredapi

In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fredapi import Fred
import datetime

plt.style.use('seaborn')
%matplotlib inline

## 2. Inflation Indicators (PCE, CPI)

In [ ]:
# Initialize FRED API (replace with your key)
fred = Fred(api_key='your_api_key_here')  

# Get inflation data
pce = fred.get_series('PCE').rename('PCE')
core_pce = fred.get_series('PCEPILFE').rename('Core PCE')
cpi = fred.get_series('CPIAUCSL').rename('CPI')

# Combine and plot
inflation_data = pd.concat([pce, core_pce, cpi], axis=1).dropna()

plt.figure(figsize=(14, 7))
for col in inflation_data.columns:
    plt.plot(inflation_data.index, inflation_data[col], label=col)
plt.title('Inflation Indicators (PCE, Core PCE, CPI)')
plt.ylabel('Index Value')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
plt.show()

## 3. Treasury Yields (2Y, 10Y, 30Y)

In [ ]:
# Get yield data
two_year = fred.get_series('DGS2')
ten_year = fred.get_series('DGS10')
thirty_year = fred.get_series('DGS30')

# Process and plot yields
yield_data = pd.concat([two_year, ten_year, thirty_year], axis=1)
yield_data.columns = ['2-Year', '10-Year', '30-Year']
yield_data = yield_data.dropna()

plt.figure(figsize=(14, 7))
for col in yield_data.columns:
    plt.plot(yield_data.index, yield_data[col], label=col)
plt.title('Treasury Yields')
plt.ylabel('Yield (%)')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
plt.show()

# Yield curve spread
yield_data['10Y-2Y Spread'] = yield_data['10-Year'] - yield_data['2-Year']
plt.figure(figsize=(14, 7))
plt.plot(yield_data.index, yield_data['10Y-2Y Spread'], label='10Y-2Y Spread', color='red')
plt.axhline(0, color='black', linestyle='--')
plt.title('10-Year minus 2-Year Treasury Yield Spread')
plt.ylabel('Spread (%)')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
plt.show()

## 4. Liquidity Indicators

In [ ]:
# Get liquidity metrics
fed_balance_sheet = fred.get_series('WALCL')
m2_money_supply = fred.get_series('M2SL')

# Combine and plot
liquidity_data = pd.concat([fed_balance_sheet, m2_money_supply], axis=1)
liquidity_data.columns = ['Fed Balance Sheet', 'M2 Money Supply']
liquidity_data = liquidity_data.dropna()

fig, ax1 = plt.subplots(figsize=(14, 7))
ax1.plot(liquidity_data.index, liquidity_data['Fed Balance Sheet'], color='blue', label='Fed Balance Sheet')
ax1.set_ylabel('Fed Balance Sheet ($B)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(liquidity_data.index, liquidity_data['M2 Money Supply'], color='green', label='M2 Money Supply')
ax2.set_ylabel('M2 Money Supply ($B)', color='green')
ax2.tick_params(axis='y', labelcolor='green')

plt.title('Liquidity Indicators')
fig.tight_layout()
plt.show()

## 5. Interactive Dashboard

In [ ]:
# Create interactive dashboard
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    subplot_titles=("Inflation Indicators", "Treasury Yields", 
                                   "Yield Spread (10Y-2Y)", "Liquidity Indicators"),
                    vertical_spacing=0.1)

# Add all data traces
for col in inflation_data.columns:
    fig.add_trace(go.Scatter(x=inflation_data.index, y=inflation_data[col], name=col), row=1, col=1)

for col in yield_data.columns[:3]:
    fig.add_trace(go.Scatter(x=yield_data.index, y=yield_data[col], name=col), row=2, col=1)

fig.add_trace(go.Scatter(x=yield_data.index, y=yield_data['10Y-2Y Spread'], 
                        name='10Y-2Y Spread', line=dict(color='red')), row=3, col=1)

fig.add_trace(go.Scatter(x=liquidity_data.index, y=liquidity_data['Fed Balance Sheet'], 
                        name='Fed Balance Sheet', line=dict(color='blue')), row=4, col=1)
fig.add_trace(go.Scatter(x=liquidity_data.index, y=liquidity_data['M2 Money Supply'], 
                        name='M2 Money Supply', line=dict(color='green')), row=4, col=1)

# Update layout
fig.update_layout(
    height=1200, 
    width=1000, 
    title_text="Market Dashboard", 
    hovermode="x unified",
    margin=dict(t=100)
)
fig.show()

## 6. Market Performance (S&P 500)

In [ ]:
# Get S&P 500 data
sp500 = yf.Ticker("^GSPC")
sp500_hist = sp500.history(period="max")

# Plot
plt.figure(figsize=(14, 7))
plt.plot(sp500_hist.index, sp500_hist['Close'])
plt.title('S&P 500 Historical Price')
plt.ylabel('Price')
plt.xlabel('Date')
plt.grid(True)
plt.show()

## Next Steps

1. Get your free FRED API key from [https://fred.stlouisfed.org/docs/api/api_key.html](https://fred.stlouisfed.org/docs/api/api_key.html)
2. Replace `your_api_key_here` with your actual key
3. Run all cells to generate the dashboard
4. Consider adding:
   - Sector performance
   - Volatility measures (VIX)
   - Economic calendar integration
   - Alert systems for threshold breaches